## Preparar datos

### Importar librerias

In [3]:
import pandas as pd

### Leer csv

In [5]:
df1 = pd.read_csv("911_sin_filtrar.csv")

### Encontrar variables que pueden servir apra predecir 

In [7]:
df1.head(10)
df1["incidente_c4"].unique()

array(['Inconsciente', 'Persona sospechosa', 'Otros',
       'Entrevista ciudadana', 'Persona extraviada', 'Escandalo',
       'Corto cables transformador', 'Casa habitacion', 'Convulsiones',
       'Solicitud de escolta', 'Apoyo vial semaforos descompuestos',
       'A policia', 'Caida de cables', 'Propiedad ajena', 'Secuestro',
       'Automovilista', 'Portacion de arma', 'Falta de agua',
       'Negocio plaza comercial mercado', 'Alarma activada',
       'Menor en riesgo', 'Intoxicado', 'A persona', 'Caida de arbol',
       'Zona sin energia electrica', 'Cable', 'Caida', 'Hechos',
       'Olor a gas combustible quimicos', 'A inmueble', 'Cilindro de gas',
       'Dolor', 'Rina', 'Consumo de droga o estupefacientes',
       'Por arma blanca', 'Quema de pirotecnia', 'Drogados', 'Ebrios',
       'Vehiculo obstruyendo', 'Por golpes', 'Franeleros', 'Infarto',
       'Robo', 'Apoyo vial embotellamiento', 'Picadura o mordedura',
       'Jugadores', 'Faltas a la moral', 'De suicidio',
      

Las variables que nos sirven son: 

Inundacion  
Encharcamiento  
Desborde canal rio presa  

#### Contar cuantos registros se tienen en total

In [10]:
df1[df1["incidente_c4"].isin([
    "Inundacion",
    "Encharcamiento",
    "Desborde canal rio presa"
])].shape[0]

9148

#### Filtrar el dataset por esas features

In [12]:
new_df = df1[df1["incidente_c4"].isin([ "Inundacion","Encharcamiento","Desborde canal rio presa"])]
new_df.head()

,folio,categoria_incidente_c4,incidente_c4,anio_creacion,mes_creacion,fecha_creacion,hora_creacion,anio_cierre,mes_cierre,fecha_cierre,hora_cierre,codigo_cierre,clas_con_f_alarma,alcaldia_cierre,colonia_cierre,manzana,latitud,longitud
1201,C5/190103/04497,Danos por fenomeno natural o tercero,Inundacion,2019,Enero,2019-01-03,14:40:55,2019,Enero,2019-01-03,14:43:04,Duplicado,EMERGENCIA,ALVARO OBREGON,SANTA MARIA NONOALCO,NaN,19.384348,-99.191522
2716,C5/190103/04003,Danos por fenomeno natural o tercero,Encharcamiento,2019,Enero,2019-01-03,13:34:14,2019,Enero,2019-01-03,13:55:35,Negativo,SERVICIO,BENITO JUAREZ,NONOALCO,NaN,19.384560,-99.190621
4009,C5/190103/04237,Danos por fenomeno natural o tercero,Inundacion,2019,Enero,2019-01-03,14:07:28,2019,Enero,2019-01-03,21:16:42,Afirmativo,EMERGENCIA,BENITO JUAREZ,NONOALCO,NaN,19.384560,-99.190621
36833,C5/190202/04894,Danos por fenomeno natural o tercero,Encharcamiento,2019,Febrero,2019-02-02,13:49:51,2019,Febrero,2019-02-02,14:13:07,Negativo,SERVICIO,BENITO JUAREZ,PORTALES IV,NaN,19.360221,-99.143410
36911,C5/190626/06427,Danos por fenomeno natural o tercero,Inundacion,2019,Junio,2019-06-26,19:33:48,2019,Junio,2019-06-27,00:14:15,Afirmativo,EMERGENCIA,TLALPAN,HEROES DE 1910,0901201241227016,19.244516,-99.240369


In [13]:
new_df.to_csv("911_clean.csv")

### Tomar solo la informacion relevante

solo los interesa, fecha, latitud, longitud, alcaldia_cierre y colonia_cierre

In [16]:
new_df = new_df[["fecha_creacion","latitud","longitud","alcaldia_cierre","colonia_cierre"]]
new_df.head()

,fecha_creacion,latitud,longitud,alcaldia_cierre,colonia_cierre
1201,2019-01-03,19.384348,-99.191522,ALVARO OBREGON,SANTA MARIA NONOALCO
2716,2019-01-03,19.384560,-99.190621,BENITO JUAREZ,NONOALCO
4009,2019-01-03,19.384560,-99.190621,BENITO JUAREZ,NONOALCO
36833,2019-02-02,19.360221,-99.143410,BENITO JUAREZ,PORTALES IV
36911,2019-06-26,19.244516,-99.240369,TLALPAN,HEROES DE 1910


#### Castear columnas relevantes para la API

In [18]:
new_df["fecha_creacion"] = pd.to_datetime(new_df["fecha_creacion"])
new_df["latitud"] = new_df["latitud"].astype(float)
new_df["longitud"] = new_df["longitud"].astype(float)
new_df.head()

,fecha_creacion,latitud,longitud,alcaldia_cierre,colonia_cierre
1201,2019-01-03,19.384348,-99.191522,ALVARO OBREGON,SANTA MARIA NONOALCO
2716,2019-01-03,19.384560,-99.190621,BENITO JUAREZ,NONOALCO
4009,2019-01-03,19.384560,-99.190621,BENITO JUAREZ,NONOALCO
36833,2019-02-02,19.360221,-99.143410,BENITO JUAREZ,PORTALES IV
36911,2019-06-26,19.244516,-99.240369,TLALPAN,HEROES DE 1910


In [30]:
new_df.shape[0]

9148

### Obtener informacion de la API para completar

In [29]:
import time
import pandas as pd
import requests


def consultar_nasa_power(lat, lon, fecha_inicio, fecha_fin):
    """Hace la petición a la API de NASA POWER para una coordenada y rango de fechas."""
    start_str = pd.to_datetime(fecha_inicio).strftime("%Y%m%d")
    end_str = pd.to_datetime(fecha_fin).strftime("%Y%m%d")

    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    parametros = "T2M,PRECTOTCORR,RH2M,WS2M,ALLSKY_SFC_SW_DWN,PS"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start": start_str,
        "end": end_str,
        "parameters": parametros,
        "community": "RE",
        "format": "JSON",
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        if response.status_code == 200:
            data = response.json()
            features = data.get("properties", {}).get("parameter", {})

            df_clima = pd.DataFrame(features)
            df_clima.index.name = "fecha_creacion"
            df_clima = df_clima.reset_index()

            df_clima["fecha_creacion"] = pd.to_datetime(
                df_clima["fecha_creacion"], format="%Y%m%d"
            )

            df_clima = df_clima.rename(
                columns={
                    "T2M": "temp",
                    "PRECTOTCORR": "precip",
                    "RH2M": "humedad",
                    "WS2M": "viento",
                    "ALLSKY_SFC_SW_DWN": "radiacion",
                    "PS": "presion",
                }
            )
            return df_clima
        else:
            print(
                f"Error API ({response.status_code}) para Lat: {lat}, Lon: {lon}"
            )
            return None
    except Exception as e:
        print(f"Error de conexión: {e}")
        return None


def enriquecer_dataset_con_clima(df_original):
    """Toma tu DataFrame original, consulta la API de la NASA de forma optimizada

    y devuelve un nuevo DataFrame con la información completa.
    """
    # Creamos una copia para no modificar el DataFrame original por referencia
    df = df_original.copy()
    df["fecha_creacion"] = pd.to_datetime(df["fecha_creacion"])

    listado_respuestas = []

    coordenadas_unicas = df.groupby(["latitud", "longitud"])

    for (lat, lon), group in coordenadas_unicas:
        fecha_min = group["fecha_creacion"].min()
        fecha_max = group["fecha_creacion"].max()

        print(f"Consultando NASA para Lat: {lat}, Lon: {lon}...")

        df_clima_punto = consultar_nasa_power(lat, lon, fecha_min, fecha_max)

        if df_clima_punto is not None:
            df_clima_punto["latitud"] = lat
            df_clima_punto["longitud"] = lon
            listado_respuestas.append(df_clima_punto)

        time.sleep(1)

    if listado_respuestas:
        df_clima_total = pd.concat(listado_respuestas, ignore_index=True)

        # Unimos el dataframe original con el climático usando las 3 columnas como llave
        df_completo = pd.merge(
            df,
            df_clima_total,
            on=["fecha_creacion", "latitud", "longitud"],
            how="left",
        )
        return df_completo
    else:
        print("No se pudieron obtener datos climáticos.")
        return df


if __name__ == "__main__":
    
    df_tu_dataset = new_df

    print("--- DataFrame Original ---")
    print(df_tu_dataset)

    # 2. Ejecutamos la función para obtener el nuevo DataFrame completo
    print("\n--- Iniciando descarga desde la NASA ---")
    df_resultado_final = enriquecer_dataset_con_clima(df_tu_dataset)

    print("\n--- Nuevo DataFrame Completo ---")
    print(df_resultado_final)

--- DataFrame Original ---
        fecha_creacion    latitud   longitud      alcaldia_cierre  \
1201        2019-01-03  19.384348 -99.191522       ALVARO OBREGON   
2716        2019-01-03  19.384560 -99.190621        BENITO JUAREZ   
4009        2019-01-03  19.384560 -99.190621        BENITO JUAREZ   
36833       2019-02-02  19.360221 -99.143410        BENITO JUAREZ   
36911       2019-06-26  19.244516 -99.240369              TLALPAN   
...                ...        ...        ...                  ...   
3596694     2021-11-20  19.454233 -99.199591       MIGUEL HIDALGO   
3600839     2021-11-20  19.244416 -99.084663           XOCHIMILCO   
3665461     2021-12-10  19.428311 -99.119893  VENUSTIANO CARRANZA   
3683740     2021-12-27  19.385491 -99.204972       ALVARO OBREGON   
3713328     2021-12-25  19.424557 -99.134986           CUAUHTEMOC   

                       colonia_cierre  
1201             SANTA MARIA NONOALCO  
2716                         NONOALCO  
4009                    

In [33]:
df_resultado_final.to_csv("final.csv",index=False)